In [11]:
import pandas as pd
import numpy as np
import sqlite3
import joblib
import os
import warnings
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)
warnings.filterwarnings("ignore")

from lightgbm import LGBMClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_val_score, train_test_split
from sklearn.metrics import (roc_auc_score, classification_report,
                              f1_score, brier_score_loss, confusion_matrix)
from sklearn.calibration import CalibratedClassifierCV
from sklearn.preprocessing import LabelEncoder, StandardScaler
from imblearn.over_sampling import SMOTE
from pytorch_tabnet.tab_model import TabNetClassifier
import shap
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

os.makedirs("../output/test_results", exist_ok=True)
print("All imports successful!")

All imports successful!


In [12]:
conn = sqlite3.connect("../data/attrition.db")
df = pd.read_sql("SELECT * FROM employees", conn)
conn.close()

# Encode
label_cols = ["BusinessTravel", "Department", "EducationField",
              "Gender", "JobRole", "MaritalStatus", "OverTime"]
le = LabelEncoder()
df_model = df.copy()
for col in label_cols:
    df_model[col] = le.fit_transform(df_model[col].astype(str))

role_avg = df_model.groupby("JobRole")["MonthlyIncome"].transform("mean")
df_model["SalaryToRoleRatio"]     = (df_model["MonthlyIncome"] / role_avg).round(3)
df_model["TenureStabilityScore"]  = (df_model["YearsAtCompany"] /
                                      (df_model["TotalWorkingYears"] + 1)).round(3)
df_model["SatisfactionComposite"] = (df_model["JobSatisfaction"] +
                                      df_model["EnvironmentSatisfaction"] +
                                      df_model["WorkLifeBalance"]) / 3

drop_cols = [c for c in ["Attrition", "Attrition_Binary", "EmployeeNumber",
                          "OverTime_Binary"] if c in df_model.columns]
X_all = df_model.drop(columns=drop_cols)
y_all = df_model["Attrition_Binary"]

# Load the best features from the pipeline
final_features = pd.read_csv("../outputs/final_features.csv", header=None)[0].tolist()
# Keep only features that exist in X_all
final_features = [f for f in final_features if f in X_all.columns]
X_all = X_all[final_features]

# 80/20 stratified split — test set locked away now
X_train, X_test, y_train, y_test = train_test_split(
    X_all, y_all, test_size=0.2, stratify=y_all, random_state=42
)

print(f"Train: {X_train.shape}  |  Test: {X_test.shape}")
print(f"Train attrition rate: {y_train.mean():.3f}")
print(f"Test attrition rate:  {y_test.mean():.3f}")

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

Train: (1176, 7)  |  Test: (294, 7)
Train attrition rate: 0.162
Test attrition rate:  0.160


In [13]:
print("Applying SMOTE to training data only...")

smote = SMOTE(random_state=42)
X_train_sm, y_train_sm = smote.fit_resample(X_train, y_train)

# Convert to numpy immediately after SMOTE — TabNet needs arrays not DataFrames
X_train_np = X_train_sm.values if hasattr(X_train_sm, 'values') else X_train_sm
y_train_np = y_train_sm.values if hasattr(y_train_sm, 'values') else y_train_sm
X_test_np  = X_test.values
y_test_np  = y_test.values

print(f"Before SMOTE: {dict(y_train.value_counts())}")
print(f"After SMOTE:  {dict(pd.Series(y_train_np).value_counts())}")

Applying SMOTE to training data only...
Before SMOTE: {0: np.int64(986), 1: np.int64(190)}
After SMOTE:  {0: np.int64(986), 1: np.int64(986)}


In [14]:
print("\n=== RETRAINING WITH SMOTE + PROPER HOLDOUT EVALUATION ===")

# LightGBM
lgbm = LGBMClassifier(
    n_estimators=300, max_depth=6, learning_rate=0.05,
    num_leaves=50, subsample=0.8, colsample_bytree=0.8,
    min_child_samples=10, class_weight="balanced",
    random_state=42, n_jobs=-1, verbose=-1
)
lgbm.fit(X_train_np, y_train_np)
lgbm_proba = lgbm.predict_proba(X_test_np)[:, 1]
results["LightGBM"] = roc_auc_score(y_test_np, lgbm_proba)
print(f"LightGBM       test AUC: {results['LightGBM']:.4f}")

# GradientBoosting
gb = GradientBoostingClassifier(
    n_estimators=200, max_depth=4, learning_rate=0.05,
    subsample=0.8, min_samples_leaf=5, random_state=42
)
gb.fit(X_train_np, y_train_np)
gb_proba = gb.predict_proba(X_test_np)[:, 1]
results["GradientBoosting"] = roc_auc_score(y_test_np, gb_proba)
print(f"GradientBoost  test AUC: {results['GradientBoosting']:.4f}")

# RandomForest
rf = RandomForestClassifier(
    n_estimators=300, max_depth=12, min_samples_leaf=2,
    class_weight="balanced", random_state=42, n_jobs=-1
)
rf.fit(X_train_np, y_train_np)
rf_proba = rf.predict_proba(X_test_np)[:, 1]
results["RandomForest"] = roc_auc_score(y_test_np, rf_proba)
print(f"RandomForest   test AUC: {results['RandomForest']:.4f}")

# Logistic Regression baseline
lr = LogisticRegression(class_weight="balanced", max_iter=1000, random_state=42)
lr.fit(X_train_np, y_train_np)
lr_proba = lr.predict_proba(X_test_np)[:, 1]
results["LogisticRegression"] = roc_auc_score(y_test_np, lr_proba)
print(f"LogisticReg    test AUC: {results['LogisticRegression']:.4f}")


=== RETRAINING WITH SMOTE + PROPER HOLDOUT EVALUATION ===
LightGBM       test AUC: 0.6674
GradientBoost  test AUC: 0.6832
RandomForest   test AUC: 0.6475
LogisticReg    test AUC: 0.7111


In [15]:
print("\n=== TABNET (FIXED) ===")

X_tr_tab = X_train_np.astype(np.float32)
y_tr_tab = y_train_np.astype(int)
X_te_tab = X_test_np.astype(np.float32)
y_te_tab = y_test_np.astype(int)

# Internal validation split
X_t2, X_v2, y_t2, y_v2 = train_test_split(
    X_tr_tab, y_tr_tab,
    test_size=0.15, stratify=y_tr_tab, random_state=42
)

tabnet = TabNetClassifier(
    n_d=32, n_a=32,
    n_steps=5,
    gamma=1.5,
    n_independent=2, n_shared=2,
    lambda_sparse=1e-4,
    mask_type="entmax",
    verbose=0, seed=42
)

tabnet.fit(
    X_t2, y_t2,                    # numpy arrays — no .values needed
    eval_set=[(X_v2, y_v2)],
    eval_metric=["auc"],
    max_epochs=200,
    patience=30,
    batch_size=128,
    virtual_batch_size=64,
    num_workers=0
)

tabnet_proba = tabnet.predict_proba(X_te_tab)[:, 1]
results["TabNet"] = roc_auc_score(y_te_tab, tabnet_proba)
print(f"TabNet (fixed) test AUC: {results['TabNet']:.4f}")


=== TABNET (FIXED) ===

Early stopping occurred at epoch 80 with best_epoch = 50 and best_val_0_auc = 0.82332
TabNet (fixed) test AUC: 0.6133


In [16]:
print("\n=== OOF BLEND ON TEST SET ===")

# Simple weighted blend using test probabilities
# Weights proportional to individual AUCs
total_auc = sum([results["LightGBM"], results["GradientBoosting"],
                 results["RandomForest"], results["TabNet"]])

w_lgbm = results["LightGBM"]   / total_auc
w_gb   = results["GradientBoosting"] / total_auc
w_rf   = results["RandomForest"] / total_auc
w_tab  = results["TabNet"]      / total_auc

blend_proba = (w_lgbm * lgbm_proba +
               w_gb   * gb_proba +
               w_rf   * rf_proba +
               w_tab  * tabnet_proba)

results["Blend (AUC-weighted)"] = roc_auc_score(y_test, blend_proba)
print(f"Blend weights — LGBM:{w_lgbm:.3f} GB:{w_gb:.3f} RF:{w_rf:.3f} TabNet:{w_tab:.3f}")
print(f"Blended test AUC: {results['Blend (AUC-weighted)']:.4f}")

# Also try equal weights
equal_blend = (lgbm_proba + gb_proba + rf_proba + tabnet_proba) / 4
results["Blend (equal)"] = roc_auc_score(y_test, equal_blend)
print(f"Equal blend AUC:  {results['Blend (equal)']:.4f}")

# Use the best blend as final
final_proba = blend_proba if results["Blend (AUC-weighted)"] >= results["Blend (equal)"] else equal_blend
best_blend_auc = max(results["Blend (AUC-weighted)"], results["Blend (equal)"])


=== OOF BLEND ON TEST SET ===
Blend weights — LGBM:0.256 GB:0.262 RF:0.248 TabNet:0.235
Blended test AUC: 0.6600
Equal blend AUC:  0.6591


In [19]:
print("\n=== SYNTHETIC DATA GENERATION + TESTING ===")

np.random.seed(99)
n_synthetic = 500

# Generate realistic synthetic employees using IBM dataset statistics
synth = pd.DataFrame({
    "Age":                  np.random.randint(22, 60, n_synthetic),
    "MonthlyIncome":        np.random.lognormal(8.4, 0.5, n_synthetic).astype(int),
    "JobSatisfaction":      np.random.choice([1,2,3,4], n_synthetic, p=[0.11,0.2,0.35,0.34]),
    "EnvironmentSatisfaction": np.random.choice([1,2,3,4], n_synthetic, p=[0.1,0.19,0.36,0.35]),
    "WorkLifeBalance": np.random.choice([1,2,3,4], n_synthetic, p=[0.05,0.23,0.61,0.11]),
    "YearsAtCompany":       np.random.exponential(7, n_synthetic).clip(0, 40).astype(int),
    "TotalWorkingYears":    np.random.exponential(11, n_synthetic).clip(0, 40).astype(int),
    "YearsInCurrentRole":   np.random.exponential(4, n_synthetic).clip(0, 18).astype(int),
    "YearsSinceLastPromotion": np.random.exponential(2, n_synthetic).clip(0, 15).astype(int),
    "JobLevel":             np.random.choice([1,2,3,4,5], n_synthetic, p=[0.26,0.33,0.2,0.14,0.07]),
    "JobRole":              np.random.randint(0, 9, n_synthetic),
    "OverTime":             np.random.choice([0,1], n_synthetic, p=[0.72, 0.28]),
    "BusinessTravel":       np.random.choice([0,1,2], n_synthetic, p=[0.7,0.19,0.11]),
    "Department":           np.random.choice([0,1,2], n_synthetic, p=[0.65,0.14,0.21]),
    "MaritalStatus":        np.random.choice([0,1,2], n_synthetic, p=[0.32,0.46,0.22]),
    "NumCompaniesWorked":   np.random.randint(0, 10, n_synthetic),
    "TrainingTimesLastYear": np.random.randint(0, 7, n_synthetic),
    "PercentSalaryHike":    np.random.randint(11, 25, n_synthetic),
    "DistanceFromHome":     np.random.randint(1, 30, n_synthetic),
    "Education":            np.random.choice([1,2,3,4,5], n_synthetic),
    "Gender":               np.random.choice([0,1], n_synthetic),
    "EducationField":       np.random.randint(0, 6, n_synthetic),
    "StockOptionLevel":     np.random.choice([0,1,2,3], n_synthetic, p=[0.47,0.36,0.12,0.05]),
    "RelationshipSatisfaction": np.random.choice([1,2,3,4], n_synthetic),
    "JobInvolvement":       np.random.choice([1,2,3,4], n_synthetic, p=[0.05,0.23,0.58,0.14]),
    "PerformanceRating":    np.random.choice([3,4], n_synthetic, p=[0.85,0.15]),
})

# Ensure TotalWorkingYears >= YearsAtCompany
synth["TotalWorkingYears"] = np.maximum(
    synth["TotalWorkingYears"], synth["YearsAtCompany"]
)

# Add derived features (same as training pipeline)
role_avg_synth = synth.groupby("JobRole")["MonthlyIncome"].transform("mean")
synth["SalaryToRoleRatio"]     = (synth["MonthlyIncome"] / role_avg_synth).round(3)
synth["TenureStabilityScore"]  = (synth["YearsAtCompany"] /
                                   (synth["TotalWorkingYears"] + 1)).round(3)
synth["SatisfactionComposite"] = (synth["JobSatisfaction"] +
                                   synth["EnvironmentSatisfaction"] +
                                   synth["WorkLifeBalance"]) / 3

# Keep only final features
synth_features = [f for f in final_features if f in synth.columns]
missing = [f for f in final_features if f not in synth.columns]
if missing:
    print(f"Adding missing features as median: {missing}")
    for f in missing:
        synth[f] = X_train[f].median()

X_synth = synth[final_features].astype(np.float32)

# Get predictions from all models
synth_lgbm   = lgbm.predict_proba(X_synth)[:, 1]
synth_gb     = gb.predict_proba(X_synth)[:, 1]
synth_rf     = rf.predict_proba(X_synth)[:, 1]
synth_tabnet = tabnet.predict_proba(X_synth.values)[:, 1]
synth_blend  = (w_lgbm*synth_lgbm + w_gb*synth_gb +
                w_rf*synth_rf + w_tab*synth_tabnet)

# Add to dataframe
synth["risk_lgbm"]   = synth_lgbm.round(4)
synth["risk_gb"]     = synth_gb.round(4)
synth["risk_rf"]     = synth_rf.round(4)
synth["risk_tabnet"] = synth_tabnet.round(4)
synth["risk_blend"]  = synth_blend.round(4)
synth["risk_tier"]   = pd.cut(synth_blend,
                               bins=[0, 0.3, 0.6, 1.0],
                               labels=["Low", "Medium", "High"])

synth.to_csv("../output/test_results/synthetic_predictions.csv", index=False)
print(f"Generated and scored {n_synthetic} synthetic employees")
print(f"\nRisk tier distribution:")
print(synth["risk_tier"].value_counts())
print(f"\nAvg blend risk score: {synth_blend.mean():.3f}")
print(f"High-risk employees:  {(synth['risk_tier']=='High').sum()}")


=== SYNTHETIC DATA GENERATION + TESTING ===
Adding missing features as median: ['DailyRate']
Generated and scored 500 synthetic employees

Risk tier distribution:
risk_tier
Low       371
Medium    119
High       10
Name: count, dtype: int64

Avg blend risk score: 0.209
High-risk employees:  10


In [21]:
print("\n=== COMPREHENSIVE MODEL COMPARISON ===")
print("(All evaluated on the same 20% holdout test set)")

all_results = pd.DataFrame.from_dict(results, orient="index", columns=["Test AUC"])
all_results = all_results.sort_values("Test AUC", ascending=False)
all_results["Source Notebook"] = [
    "05_Test", "05_Test", "05_Test", "05_Test",
    "05_Test (fixed)", "05_Test", "05_Test"
][:len(all_results)]

print(all_results.round(4).to_string())
all_results.to_csv("../output/test_results/model_comparison_all.csv")

# Visualise
fig, ax = plt.subplots(figsize=(10, 5))
colors = ["#534AB7" if v == all_results["Test AUC"].max()
          else "#D3D1C7" for v in all_results["Test AUC"]]
bars = ax.barh(all_results.index, all_results["Test AUC"], color=colors)
ax.axvline(0.5, color="red", ls="--", alpha=0.4, label="Random (0.5)")
ax.axvline(0.8, color="green", ls="--", alpha=0.4, label="Good (0.8)")
for bar, val in zip(bars, all_results["Test AUC"]):
    ax.text(bar.get_width() + 0.002, bar.get_y() + bar.get_height()/2,
            f"{val:.4f}", va="center", fontsize=11)
ax.set_xlabel("Test AUC (holdout)")
ax.set_title("Model comparison — same holdout test set")
ax.legend(); ax.grid(True, alpha=0.2, axis="x")
ax.set_xlim(0.4, 1.0)
plt.tight_layout()
plt.savefig("../output/test_results/model_comparison_chart.png", dpi=150, bbox_inches="tight")
plt.close()
print("Saved model_comparison_chart.png")


=== COMPREHENSIVE MODEL COMPARISON ===
(All evaluated on the same 20% holdout test set)
                      Test AUC  Source Notebook
LogisticRegression      0.7111          05_Test
GradientBoosting        0.6832          05_Test
LightGBM                0.6674          05_Test
Blend (AUC-weighted)    0.6600          05_Test
Blend (equal)           0.6591  05_Test (fixed)
RandomForest            0.6475          05_Test
TabNet                  0.6133          05_Test
Saved model_comparison_chart.png


In [23]:
print("\n=== DETAILED TEST REPORT — BEST MODEL ===")

best_model_name = all_results["Test AUC"].idxmax()
best_proba = {
    "LightGBM": lgbm_proba,
    "GradientBoosting": gb_proba,
    "RandomForest": rf_proba,
    "TabNet (fixed)": tabnet_proba,
    "Blend (AUC-weighted)": blend_proba,
    "Blend (equal)": equal_blend,
}.get(best_model_name, blend_proba)

THRESHOLD = 0.25   # business-optimized (low threshold catches more attritors)
y_pred_final = (best_proba >= THRESHOLD).astype(int)

print(f"Best model: {best_model_name}")
print(f"Test AUC:   {roc_auc_score(y_test, best_proba):.4f}")
print(f"Threshold:  {THRESHOLD}")
print(f"\nClassification Report:")
print(classification_report(y_test, y_pred_final,
                            target_names=["Stayed", "Left"]))

# Confusion matrix
cm = confusion_matrix(y_test, y_pred_final)
fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(cm, cmap="Blues")
ax.set_xticks([0,1]); ax.set_yticks([0,1])
ax.set_xticklabels(["Stayed", "Left"])
ax.set_yticklabels(["Stayed", "Left"])
ax.set_xlabel("Predicted"); ax.set_ylabel("Actual")
ax.set_title(f"Confusion matrix — {best_model_name}")
for i in range(2):
    for j in range(2):
        ax.text(j, i, cm[i,j], ha="center", va="center",
                color="white" if cm[i,j] > cm.max()/2 else "black",
                fontsize=14, fontweight="bold")
plt.colorbar(im)
plt.tight_layout()
plt.savefig("../output/test_results/confusion_matrix.png", dpi=150, bbox_inches="tight")
plt.close()
print("Saved confusion_matrix.png")

# Risk distribution on test set
test_results_df = X_test.copy()
test_results_df["actual"] = y_test.values
test_results_df["predicted_prob"] = best_proba
test_results_df["predicted_label"] = y_pred_final
test_results_df["risk_tier"] = pd.cut(best_proba,
                                       bins=[0, 0.3, 0.6, 1.0],
                                       labels=["Low", "Medium", "High"])
test_results_df.to_csv("../output/test_results/test_set_predictions.csv", index=False)
print("Saved test_set_predictions.csv")


=== DETAILED TEST REPORT — BEST MODEL ===
Best model: LogisticRegression
Test AUC:   0.6600
Threshold:  0.25

Classification Report:
              precision    recall  f1-score   support

      Stayed       0.88      0.64      0.74       247
        Left       0.22      0.55      0.32        47

    accuracy                           0.62       294
   macro avg       0.55      0.59      0.53       294
weighted avg       0.78      0.62      0.67       294

Saved confusion_matrix.png
Saved test_set_predictions.csv


In [24]:
print("\n" + "="*60)
print("FULL PROJECT MODEL SUMMARY")
print("="*60)

summary = {
    "Notebook 02 — Basic RF":          {"AUC": "~0.85 (train)", "Note": "Overfit — measured on training data"},
    "Notebook 02b — Stacking (LGBM+GB+RF)": {"AUC": "~0.88 (train)", "Note": "Overfit — measured on training data"},
    "Notebook 04 — OOF Pipeline":      {"AUC": "0.7744 (OOF)",  "Note": "Honest — out-of-fold on training data"},
    "Notebook 05 — Holdout Test":      {"AUC": f"{best_blend_auc:.4f} (holdout)", "Note": "Most honest — never seen during training"},
}

for nb, info in summary.items():
    print(f"\n{nb}")
    print(f"  AUC:  {info['AUC']}")
    print(f"  Note: {info['Note']}")

print(f"\nSynthetic data test:")
print(f"  500 employees scored successfully")
print(f"  High-risk identified: {(synth['risk_tier']=='High').sum()}")
print(f"  Saved to: outputs/test_results/synthetic_predictions.csv")

print(f"\nOutput files in outputs/test_results/:")
for f in sorted(os.listdir("../output/test_results")):
    print(f"  {f}")


FULL PROJECT MODEL SUMMARY

Notebook 02 — Basic RF
  AUC:  ~0.85 (train)
  Note: Overfit — measured on training data

Notebook 02b — Stacking (LGBM+GB+RF)
  AUC:  ~0.88 (train)
  Note: Overfit — measured on training data

Notebook 04 — OOF Pipeline
  AUC:  0.7744 (OOF)
  Note: Honest — out-of-fold on training data

Notebook 05 — Holdout Test
  AUC:  0.6600 (holdout)
  Note: Most honest — never seen during training

Synthetic data test:
  500 employees scored successfully
  High-risk identified: 10
  Saved to: outputs/test_results/synthetic_predictions.csv

Output files in outputs/test_results/:
  confusion_matrix.png
  model_comparison_all.csv
  model_comparison_chart.png
  synthetic_predictions.csv
  test_set_predictions.csv
